In [15]:
import json
import os
import sys
from pathlib import Path
from typing import Any, TypeVar

import pydantic
from dotenv import load_dotenv
from openai import OpenAI
from tqdm import tqdm

from icecream import ic

from PydanticContracts import (
    SyntheticChunkingExample,
    GeneralJudgeResult,
    SizeComplianceJudgeResult,
    IntrachunkCohesionJudgeResult,
    ContextualCoherenceJudgeResult,
    ChunkScoreJudgeResult,
    HopeConceptUnityJudgeResult,
    HopeSemanticIndependenceJudgeResult,
    HopeInformationPreservationJudgeResult
)

ChecksT = TypeVar("ChecksT", bound=pydantic.BaseModel)
ResultT = TypeVar("ResultT", bound=pydantic.BaseModel)

load_dotenv()

### Generator

MODEL_NAME = "deepseek-v4-pro"
BASE_URL = "https://api.deepseek.com"
TEMPERATURE = 1.0
REASONING = False
REASONING_EFFORT = "medium"
MAX_TOKENS = (8192, 10000)[REASONING]
TIMEOUT_SECONDS = 240.0
PAIRS_PER_PROMPT = 3
REGENERATION_ATTEMPTS = 3

### Judge
JUDGE_MODEL_NAME = "deepseek-v4-pro"
JUDGE_BASE_URL = "https://api.deepseek.com"
JUDGE_TEMPERATURE = 0.8
JUDGE_REASONING = True
JUDGE_REASONING_EFFORT = "medium"
JUDGE_MAX_TOKENS = (8192, 16000)[REASONING]
JUDGE_TIMEOUT_SECONDS = 240.0
JUDGE_REGENERATION_ATTEMPTS = 3

cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd if (cwd / "prompts").is_dir() else cwd.parent
sys.path.insert(0, str(PROJECT_ROOT))

PROMPTS_ROOT = PROJECT_ROOT / "prompts"
OUTPUT_ROOT = PROJECT_ROOT / "data" / "generated"
SELECTED_PROMPTS = [
    (Path("general_validation.md"), GeneralJudgeResult),
    # (Path("metrics/size_compliance.md"), SizeComplianceJudgeResult),
    (Path("metrics/intrachunk_cohesion.md"), IntrachunkCohesionJudgeResult),
    (Path("metrics/contextual_coherence.md"), SizeComplianceJudgeResult),
    (Path("metrics/boundary_clarity.md"), ContextualCoherenceJudgeResult),
    (Path("metrics/chunk_score.md"), ChunkScoreJudgeResult),
    (Path("metrics/hope_concept_unity.md"), HopeConceptUnityJudgeResult), 
    (Path("metrics/hope_semantic_independence.md"), HopeSemanticIndependenceJudgeResult),
    (Path("metrics/hope_information_preservation.md"), HopeInformationPreservationJudgeResult),
]

ic(SELECTED_PROMPTS)

client = OpenAI(
    api_key=os.environ["API_KEY"],
    base_url=BASE_URL,
    timeout=TIMEOUT_SECONDS,
)

ic| SELECTED_PROMPTS: [(PosixPath('general_validation.md'),
                        <class 'PydanticContracts.GeneralJudgeResult'>),
                       (PosixPath('metrics/intrachunk_cohesion.md'),
                        <class 'PydanticContracts.IntrachunkCohesionJudgeResult'>),
                       (PosixPath('metrics/contextual_coherence.md'),
                        <class 'PydanticContracts.SizeComplianceJudgeResult'>),
                       (PosixPath('metrics/boundary_clarity.md'),
                        <class 'PydanticContracts.ContextualCoherenceJudgeResult'>),
                       (PosixPath('metrics/chunk_score.md'),
                        <class 'PydanticContracts.ChunkScoreJudgeResult'>),
                       (PosixPath('metrics/hope_concept_unity.md'),
                        <class 'PydanticContracts.HopeConceptUnityJudgeResult'>),
                       (PosixPath('metrics/hope_semantic_independence.md'),
                        <class 'PydanticContracts.

In [16]:
def save_json(data: dict[str, Any] | list[dict[str, Any]], path: Path) -> Path:
    """Save JSON objects in a human-readable UTF-8 file."""
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(
        json.dumps(data, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )
    return path

In [17]:
SyntheticChunkingExample.model_json_schema()

{'$defs': {'ChunkingVariant': {'additionalProperties': False,
   'properties': {'chunks': {'items': {'type': 'string'},
     'minItems': 1,
     'title': 'Chunks',
     'type': 'array'},
    'rationale': {'minLength': 1, 'title': 'Rationale', 'type': 'string'},
    'focus': {'additionalProperties': True,
     'title': 'Focus',
     'type': 'object'}},
   'required': ['chunks', 'rationale'],
   'title': 'ChunkingVariant',
   'type': 'object'}},
 'additionalProperties': False,
 'properties': {'document_title': {'minLength': 1,
   'title': 'Document Title',
   'type': 'string'},
  'source_document': {'minLength': 1,
   'title': 'Source Document',
   'type': 'string'},
  'positive': {'$ref': '#/$defs/ChunkingVariant'},
  'negative': {'$ref': '#/$defs/ChunkingVariant'},
  'controlled_change': {'minLength': 1,
   'title': 'Controlled Change',
   'type': 'string'},
  'expected_relation': {'const': 'positive_higher_than_negative',
   'title': 'Expected Relation',
   'type': 'string'}},
 'requi

In [ ]:
def llm_judge(
    example: SyntheticChunkingExample,
    system_prompt: str,
    metric_prompt: str,
    result_model: type[ResultT],
) -> ResultT:
    messages = [
        {
            "role": "system",
            "content": system_prompt,
        },
        {
            "role": "user",
            "content": (
                f"{metric_prompt}\n\n"
                "Проверь следующий синтетический пример:\n\n"
                f"{example.model_dump_json(indent=2)}"
            ),
        },
    ]

    result = None

    for attempt in range(JUDGE_REGENERATION_ATTEMPTS):
        try:
            response = client.chat.completions.create(
                model=JUDGE_MODEL_NAME,
                messages=messages,
                temperature=JUDGE_TEMPERATURE,
                max_tokens=JUDGE_MAX_TOKENS,
                response_format={"type": "json_object"},
                extra_body={"thinking": {"type": ("disabled", "enabled")[JUDGE_REASONING]}},
                reasoning_effort=JUDGE_REASONING_EFFORT,
            )
            content = response.choices[0].message.content
            ic(response.choices[0].message.reasoning_content)
            ic(content)
            result = result_model.model_validate_json(content)
            break
        except pydantic.ValidationError:
            print("Retrying judging..")
            continue

    return result

In [13]:
def generate(
    system_prompt: str,
    judge_system_prompt: str,
    user_prompt: str,
    judge_metric_prompt: str,
    judge_result_model: type[ResultT],
):
    result = None
    for attempt in range(REGENERATION_ATTEMPTS):
        response = client.chat.completions.create(
            model=MODEL_NAME,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt},
            ],
            temperature=TEMPERATURE,
            max_tokens=MAX_TOKENS,
            response_format={"type": "json_object"},
            extra_body={"thinking": {"type": ("disabled", "enabled")[REASONING]}},
            reasoning_effort=REASONING_EFFORT,
        )
        content = response.choices[0].message.content
        try:
            result = SyntheticChunkingExample.model_validate_json(content)

            print("Sending to judge..")

            judge_verdict = llm_judge(
                example=result,
                system_prompt=judge_system_prompt,
                metric_prompt=judge_metric_prompt,
                result_model=judge_result_model
            )

            ic(judge_verdict)

            break
        except pydantic.ValidationError:
            tqdm.write("Retrying..")
    return result.model_dump()

In [14]:
system_prompt = (PROMPTS_ROOT / "system.md").read_text(encoding="utf-8")
judge_system_prompt = (PROMPTS_ROOT / "judge" / "system.md").read_text(encoding="utf-8")

for prompt_path, judge_result_model in tqdm(SELECTED_PROMPTS, desc="Prompts", position=0):
    prompt_name = prompt_path.stem
    user_prompt = (PROMPTS_ROOT / prompt_path).read_text(encoding="utf-8")
    judge_metric_prompt = (PROMPTS_ROOT / 'judge' / prompt_path).read_text(encoding="utf-8")
    results = []
    output_path = ""
    for pair_number in tqdm(
        range(1, PAIRS_PER_PROMPT + 1), desc="Items", position=1, leave=False
    ):
        result = generate(
            system_prompt=system_prompt,
            judge_system_prompt=judge_system_prompt,
            user_prompt=user_prompt,
            judge_metric_prompt=None,
            judge_result_model=judge_result_model,
        )

        results.append(result)

        output_path = save_json(results, OUTPUT_ROOT / f"{prompt_name}.json")
    tqdm.write(f"Saved {output_path}")

Prompts:   0%|          | 0/8 [00:00<?, ?it/s]

Sending to judge..


ic| response: ChatCompletion(id='1ce45a22-522c-4134-860b-6aff5ce84b24', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='{
                "valid": true,
                "severity": "minor",
                "quality_score": 4,
                "reason": "Граница в negative действительно смещена внутрь пункта 1.2, разрывая фразу «Полное наименование Организации на русском языке»; positive не содержит такого разрыва. Изменение локально и содержит целевое нарушение. Однако заявление об OCR-дефекте ошибочно: строчная «на» исходно была в тексте, а не заменена с «На», поэтому описание контролируемого изменения неточно."
              }', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None, reasoning_content='We need answer in JSON. Need evaluate synthetic pair. Need understand schema? User says return JSON strictly from specialized prompt but no schema provided. We need infer likely fields: valid

Retrying judging..


Prompts:   0%|          | 0/8 [01:49<?, ?it/s]


KeyboardInterrupt: 